<a href="https://colab.research.google.com/github/001123/test-google-colab/blob/main/quickstarts/Get_started_managed_agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##### Copyright 2026 Google LLC.

In [58]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini Agents API: Build managed agents with the Interactions API

<a class="tfo-notebook-buttons" target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Get_started_managed_agents.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

The [Interactions API](https://ai.google.dev/gemini-api/docs/interactions) provides a unified interface for working with Gemini models and agents. The [Getting Started notebook](./Get_started_interactions_api.ipynb) covers how to use it with standard Gemini **models** for text generation, multi-turn conversations, and tool use.

This notebook focuses on something different: **managed agents** with the `antigravity-preview-05-2026` agent.

### `agent=` vs `model=`

When you call the Interactions API, you choose between two modes:

| Parameter | What runs | Best for |
|-----------|-----------|----------|
| `model="gemini-..."` | A standard Gemini model | Text generation, structured output, function calling |
| `agent="antigravity-preview-05-2026"` | A **managed agent** in a sandboxed Linux environment | Autonomous tasks: code execution, web research, file management |

With `model=`, you get a stateless LLM call (see the [Getting Started notebook](./Get_started_interactions_api.ipynb)). With `agent=`, you spin up an autonomous agent that can **reason, plan, write and execute code, browse the web, and manage files** — all inside a secure sandbox, without you writing any orchestration logic.

This notebook walks you through the agent mode step by step:

1. **Simple questions** — use the agent like an LLM (it works, but it's overkill!)
2. **Multi-turn conversations** — persistent sandbox = built-in memory
3. **Using tools** — code execution, web search, file operations
4. **Loading data into the sandbox** — inject files before the agent starts
5. **Creating reusable custom agents** — bundle instructions, skills, and environment

<a name="setup"></a>
## Setup

### Install SDK

Install the SDK from [PyPI](https://github.com/googleapis/python-genai). It's recommended to always use the latest version.

In [59]:
%pip install -U -q "google-genai>=2.9.0"

### Setup your API key

To run the following cell, your API key must be stored it in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key or you aren't sure how to create a Colab Secret, see [Authentication ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../quickstarts/Authentication.ipynb) for an example.

In [60]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

### Initialize SDK client

With the new SDK, now you only need to initialize a client with you API key.

In [61]:
import uuid
from google import genai
from google.genai import types
from IPython.display import Markdown

client = genai.Client(api_key=GEMINI_API_KEY)

# The default managed agent.
AGENT = "antigravity-preview-05-2026"

# Generate a unique suffix for this notebook session to prevent agent ID conflicts
UNIQUE_SUFFIX = uuid.uuid4().hex[:8]

print("Client ready!")

Client ready!


## 1. Simple questions — the agent as an LLM

The simplest way to use a managed agent is to ask it a question, just like you'd call a standard Gemini model. Pass `agent="antigravity-preview-05-2026"` and `environment="remote"` to create a fresh Linux sandbox for the agent.

This works, but it's a bit like driving a Formula 1 car to the grocery store — the agent has code execution, web search, and file management capabilities that are all sitting idle for a simple factual question.

In [62]:
interaction = client.interactions.create(
    agent=AGENT,
    input="What is the capital of France?",
    environment="remote",
)

Markdown(interaction.output_text)

The capital of France is **Paris**.

In [63]:
# The response also includes metadata about the agent's sandbox.
print(f"Status:         {interaction.status}")
print(f"Interaction ID: {interaction.id}")
print(f"Environment ID: {interaction.environment_id}")

Status:         completed
Interaction ID: v1_ChdwR2F6YXBhZ0R1TDQtOFlQMHNyNDBBSRIXcEdhemFwYWdEdUw0LThZUDBzcjQwQUk
Environment ID: 95f59a06667c428e02a7dbbc02dd417b


Notice the `environment_id` in the response. That's the agent's persistent Linux sandbox. Even for this simple question, a full container was provisioned. Let's make use of that persistence next.

## 2. Multi-turn conversations

Since each agent runs in a persistent sandbox, you can **continue where you left off** by reusing the `environment_id` and linking turns with `previous_interaction_id`.

This is fundamentally different from stateless `model=` calls. The agent has a true *persistent environment* — files it creates stick around, packages it installs remain available, and conversation context is preserved.

In [64]:
# Turn 1: Introduce yourself.
turn1 = client.interactions.create(
    agent=AGENT,
    input="Hi! My name is Alice and I'm a software engineer. Remember that in a knowledge.md doc.",
    environment= "remote",
)

Markdown(f"**Turn 1:** {turn1.output_text}")

**Turn 1:** Nice to meet you, Alice! I have created `knowledge.md` and recorded that your name is Alice and you are a software engineer.

In [65]:
# Turn 2: Ask whether the agent remembers.
# Pass environment_id and previous_interaction_id to continue the conversation.
turn2 = client.interactions.create(
    agent=AGENT,
    input="what's my name and what do I do?",
    environment= turn1.environment_id,
)

Markdown(f"**Turn 2:** {turn2.output_text}")

**Turn 2:** Based on `knowledge.md`:

- **Name:** Alice
- **Occupation:** Software Engineer

The agent remembered across turns because you passed `environment` with the previous environment ID — same sandbox, which means same files.

You could have achieved the same result using `previous_interaction_id` to keep the history of the previous conversation, but that would not have showcased the environement specificities.

This is how you build stateful, multi-turn workflows. See the [Getting Started notebook](./Get_started_interactions_api.ipynb) for `model=`-based multi-turn using `previous_interaction_id` alone (without environments).

## 3. Using tools — where the agent shines

This is where managed agents go beyond a standard chat model. The antigravity-preview-05-2026 agent has **built-in tools** it uses autonomously — you don't declare them, just describe your goal and the agent figures out what to use.

| Tool | Description |
|------|-------------|
| `bash` | Execute shell commands in the sandbox |
| `google_search` | Search the web for current information |
| `url_context` | Fetch and extract text from URLs |
| `write_file` | Create or overwrite files in the sandbox |
| `read_file` | Read file contents from the sandbox |
| `list_files` | List directory contents |
| `delete_file` | Remove files from the sandbox |

For the standard `model=`-based tools (Google Search grounding, code execution, function calling), see the [Getting Started notebook](./Get_started_interactions_api.ipynb) and the dedicated tool notebooks:
- [Code Execution](./Code_Execution.ipynb)
- [Search Grounding](./Search_Grounding.ipynb)
- [Function Calling](./Function_calling.ipynb)

### Code execution

Ask a computational question and the agent will write code, run it in its sandbox, and return the verified result.

In [66]:
interaction = client.interactions.create(
    agent=AGENT,
    input=(
        "Write a Python script that computes the first 20 Fibonacci numbers. "
        "Run it and show the output."
    ),
    environment="remote",
)

Markdown(interaction.output_text)

I have created `fibonacci.py` to generate the first 20 Fibonacci numbers and executed it.

### Script (`fibonacci.py`)

```python
def fibonacci(n):
    fib_series = []
    a, b = 0, 1
    for _ in range(n):
        fib_series.append(a)
        a, b = b, a + b
    return fib_series

if __name__ == "__main__":
    n = 20
    fib_numbers = fibonacci(n)
    print(f"First {n} Fibonacci numbers:")
    for i, num in enumerate(fib_numbers, 1):
        print(f"{i}: {num}")
```

### Execution Output

```text
First 20 Fibonacci numbers:
1: 0
2: 1
3: 1
4: 2
5: 3
6: 5
7: 8
8: 13
9: 21
10: 34
11: 55
12: 89
13: 144
14: 233
15: 377
16: 610
17: 987
18: 1597
19: 2584
20: 4181
```

### Inspecting steps — what the agent actually did

The `steps` field in the response shows the agent's reasoning chain: its thoughts, tool calls, tool results, and final output. This is useful for debugging and understanding the agent's behavior.

In [67]:
# Inspect the steps from the Fibonacci interaction above.
for i, step in enumerate(interaction.steps):
    step_type = step.type
    print(f"--- Step {i} [{step_type}] ---")

    # Tool call steps show which tool was invoked and with what arguments.
    if hasattr(step, "name") and step.name:
        print(f"  Tool: {step.name}")
        if hasattr(step, "arguments"):
            args_str = str(step.arguments)[:300]
            print(f"  Args: {args_str}")

    # Content steps contain the agent's text output.
    if hasattr(step, "content") and step.content:
        for c in step.content:
            if hasattr(c, "text"):
                print(f"  Text: {c.text[:300]}")
    print()

--- Step 0 [thought] ---

--- Step 1 [function_call] ---
  Tool: write_file
  Args: {'toolSummary': 'Fibonacci script creation', 'toolAction': 'Creating Fibonacci script', 'explanation': 'Created fibonacci.py to compute the first 20 Fibonacci numbers.', 'path': 'fibonacci.py', 'content': 'def fibonacci(n):\n    fib_series = []\n    a, b = 0, 1\n    for _ in range(n):\n        fib_s

--- Step 2 [function_result] ---
  Tool: write_file

--- Step 3 [code_execution_call] ---

--- Step 4 [code_execution_result] ---

--- Step 5 [model_output] ---
  Text: I have created `fibonacci.py` to generate the first 20 Fibonacci numbers and executed it.

### Script (`fibonacci.py`)

```python
def fibonacci(n):
    fib_series = []
    a, b = 0, 1
    for _ in range(n):
        fib_series.append(a)
        a, b = b, a + b
    return fib_series

if __name__ == "_



### Web search

The agent can search the web autonomously when it needs up-to-date information.

In [68]:
interaction = client.interactions.create(
    agent=AGENT,
    input="What were the top 3 news stories about Google this week? Summarize them briefly.",
    environment="remote",
)

Markdown(interaction.output_text)

Here are the top three news stories regarding Google for the week of September 16–23, 2026:

---

### 1. Unsealed Ad Tech Antitrust Remedies: Google Dodges Breakup but Faces Oversight
* **Summary:** A federal court in Virginia unsealed a comprehensive 106-page remedies opinion in the Department of Justice’s advertising technology monopolization case against Google (*United States et al. v. Google LLC*) [1.1, 1.9]. While U.S. District Judge Leonie Brinkema rejected the DOJ’s push to force a divestiture or structural breakup of Google’s AdX ad exchange and ad server (DFP) [1.3, 1.8], she ordered sweeping behavioral remedies [1.1]. Google must operate under an independent antitrust compliance monitor for six years, adhere to strict interoperability and data-sharing standards, and eliminate self-preferencing bidding rules between AdWords and Google ad tools [1.8, 1.17]. Both parties will submit a joint final judgment, and Google confirmed plans to appeal the underlying liability ruling [1.6, 1.9].

---

### 2. EU Fines Google €403 Million ($463M) Over Location Data Practices
* **Summary:** Ireland’s Data Protection Commission (DPC)—Google’s lead data privacy regulator in the European Union—concluded a six-year inquiry by slapping Google with a €403 million penalty for violating the General Data Protection Regulation (GDPR) [2.1, 2.2]. The investigation found that between May 2018 and February 2020, Google failed to process users' location data in a lawful, fair, and transparent manner across three features: **Web & App Activity**, **Location History**, and Android’s **Location Accuracy** [2.1, 2.2]. The regulator determined that users were not adequately informed that location information was used to target advertising and infer personal profiles, and that Google retained location data longer than necessary [2.2, 2.6]. Google was also given six months to bring its data processing practices into compliance [2.1, 2.2].

---

### 3. Federal Antitrust Lawsuit Accuses Google and AI Rivals of "AI Slowdown" Pact
* **Summary:** A proposed class-action lawsuit was filed in the U.S. District Court for the Northern District of California against Google, OpenAI, Anthropic, and SpaceXAI [3.3, 3.4]. Brought by paying subscribers of generative AI tools (Gemini, ChatGPT, Claude, and Grok), the complaint accuses the four competing AI leaders of violating Section 1 of the Sherman Antitrust Act by coordinating an agreement to artificially slow the pace of AI development under the premise of "safety" [3.3, 3.9]. The suit focuses on events following a September 12 essay by Anthropic CEO Dario Amodei proposing industrywide coordination to pace frontier models, which was endorsed publicly that same day by Google DeepMind head Demis Hassabis, OpenAI CEO Sam Altman, and SpaceXAI/xAI CEO Elon Musk [3.3, 3.4]. The plaintiffs argue that private agreements between competitors to suppress product capabilities diminish consumer value and constitute unlawful market restraint [3.3, 3.6].

---

### Sources
- [1.1] [DOJ: Department of Justice Again Wins Substantial Relief Against Google](https://www.justice.gov/opa/pr/department-justice-again-wins-substantial-relief-against-google)
- [1.3] [MoginLaw: Google Keeps AdX as Court Orders Behavioral Remedies in Ad-Tech Antitrust Case](https://moginlawllp.com/google-keeps-adx-as-court-orders-behavioral-remedies-in-ad-tech-antitrust-case/)
- [1.6] [Courthouse News Service: Google dodges antitrust breakup of ad tech business](https://www.courthousenews.com/google-dodges-antitrust-breakup-of-ad-tech-business/)
- [1.8] [Search Engine Land: Google faces six years of court oversight in ad tech antitrust case](https://searchengineland.com/google-six-years-oversight-ad-tech-antitrust-case-489201)
- [1.9] [Reuters / WTVB: Google should relax ad tech rules, appoint antitrust monitor, US judge finds](https://wtvbam.com/2026/09/16/google-should-appoint-antitrust-compliance-officer-us-judge-says-in-ad-tech-case/)
- [1.17] [DOJ Press Release Details: Remedial Measures](https://www.justice.gov/opa/pr/department-justice-again-wins-substantial-relief-against-google)
- [2.1] [The Hacker News: Google Fined €403 Million Over GDPR Violations Tied to Location Data](https://thehackernews.com/2026/09/google-fined-403-million-over-gdpr.html)
- [2.2] [Data Protection Commission Ireland: Inquiry into Google's processing of location data](https://www.dataprotection.ie/en/news-media/latest-news/data-protection-commission-fines-google-eu403-million-following-inquiry-googles-processing-location)
- [2.6] [News On AIR: Google fined 403 million euros by Ireland over unlawful tracking](https://newsonair.gov.in/google-fined-403-million-euros-by-ireland-over-unlawful-tracking-of-users-location-data/)
- [3.3] [Quartz: Antitrust lawsuit targets Anthropic, OpenAI, Google, SpaceXAI AI slowdown](https://qz.com/antitrust-lawsuit-anthropic-openai-google-spacexai-ai-slowdown-092026)
- [3.4] [CBS News: Lawsuit says Anthropic, OpenAI, SpaceXAI and Google made illegal deal on AI slowdown](https://www.cbsnews.com/news/ai-slowdown-lawsuit-openai-anthropic-google/)
- [3.6] [The Independent: Lawsuit accuses Anthropic, OpenAI, Google and SpaceXAI of illegal agreement to slow AI](https://www.independent.co.uk/news/world/americas/google-anthropic-openai-spacexai-lawsuit-antitrust-laws-b3052994.html)
- [3.9] [Enterprise DNA: Big AI's Safety Pact Triggers Federal Antitrust Lawsuit](https://enterprisedna.co/resources/news/ai-antitrust-lawsuit-openai-anthropic-google-slowdown-2026/)

### File operations

The agent can create, read, and manage files in its sandbox. Files persist within the environment across turns.

In [69]:
# Ask the agent to create a file, run it, and show results.
interaction = client.interactions.create(
    agent=AGENT,
    input=(
        "Create a Python file called 'analysis.py' that generates 50 random numbers, "
        "computes mean, median, and standard deviation, then prints the results. "
        "Run it and show the output."
    ),
    environment="remote",
)

Markdown(interaction.output_text)

The Python script `analysis.py` has been created and executed.

### File Contents (`analysis.py`)

```python
import random
import statistics

# Generate 50 random numbers between 1 and 100
numbers = [random.uniform(1, 100) for _ in range(50)]

# Compute statistics
mean_val = statistics.mean(numbers)
median_val = statistics.median(numbers)
stdev_val = statistics.stdev(numbers)

# Print results
print(f"Sample size: {len(numbers)}")
print(f"Mean: {mean_val:.4f}")
print(f"Median: {median_val:.4f}")
print(f"Standard Deviation: {stdev_val:.4f}")
```

### Execution Output

```text
Sample size: 50
Mean: 51.9800
Median: 55.0787
Standard Deviation: 27.9404
```

## 4. Loading data into the agent's sandbox

You can inject files into the agent's environment **before it starts** using `sources`. This is how you provide data, configuration, or code for the agent to work with.

| Source type | Description | Best for |
|------------|-------------|----------|
| `inline` | Embed content directly (max 75 KB) | Config files, small scripts |
| `gcs` | Load from Google Cloud Storage | Large datasets |
| `repository` | Load from GitHub | Code repositories |

In [70]:
# Inject a CSV file inline and ask the agent to analyze it.
csv_data = """name,age,city,score
              Alice,28,Paris,92
              Bob,35,London,87
              Charlie,42,Berlin,95
              Diana,31,Tokyo,88
              Eve,26,Sydney,91"""

interaction = client.interactions.create(
    agent=AGENT,
    input="Read the file data.csv, analyze it, and tell me who scored the highest.",
    environment={
        "type": "remote",
        "sources": [
            {
                "type": "inline",
                "content": csv_data,
                "target": "/workspace/data.csv",
            }
        ],
    },
)

Markdown(interaction.output_text)

Based on the analysis of `data.csv`, **Charlie** scored the highest.

### Score Summary:
- **Charlie**: 95
- **Alice**: 92
- **Eve**: 91
- **Diana**: 88
- **Bob**: 87

You can also load from other sources:

```python
# From Google Cloud Storage
{"type": "gcs", "source": "gs://my-bucket/data/", "target": "/workspace/data/"}

# From a GitHub repository
{"type": "repository", "source": "https://github.com/user/repo", "target": "/workspace/repo/"}
```

You can combine multiple sources in a single request — the agent will have access to all of them at startup.

**Pro tip:** You can use that to add skills to you agent, as you'll see next.

## 5. Creating reusable custom agents

So far, every interaction has used the base `antigravity-preview-05-2026` agent with inline instructions. Once you've found a setup that works well, you can **persist it into a named custom agent** that bundles:

- **Instructions** — system prompt that defines the agent's behavior
- **Environment** — pre-configured sandbox with files and sources
- **Skills** — `SKILL.md` files that teach the agent specialized capabilities

This is the recommended workflow:
1. **Prototype** with `agent="antigravity-preview-05-2026"` — iterate on instructions, sources, and prompts
2. **Create** a named agent via the `/agents` endpoint
3. **Invoke** your agent by name from any client

### Creating a custom agent

In [71]:
# Create a custom data analysis agent using the SDK.
my_agent = client.agents.create(
    id=f"my-data-analyst-{UNIQUE_SUFFIX}",
    base_agent=AGENT,
    system_instruction=(
        "You are a data analysis assistant. "
        "Always write Python code using pandas to answer questions. "
        "Show your code and output clearly. "
        "When creating visualizations, save them as PNG files."
    ),
    base_environment={
        "type": "remote",
    },
)

print(f"✓ Agent created: {my_agent.id}")

✓ Agent created: my-data-analyst-915849b1


### Using a custom agent

Once created, invoke your agent by name. It will follow its instructions automatically.

In [72]:
# Invoke the custom agent.
interaction = client.interactions.create(
    agent=f"my-data-analyst-{UNIQUE_SUFFIX}",
    input=(
        "Generate a sample dataset of 100 sales records with columns: "
        "product, region, revenue, quantity. "
        "Find the top 5 products by total revenue and show the analysis."
    ),
    environment="remote",
)

Markdown(interaction.output_text)

Here is the complete Python solution using `pandas` to generate the dataset, analyze the top 5 products by total revenue, and create visualizations.

---

### 1. Python Code

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set random seed for reproducibility
np.random.seed(42)

# Define products and regions
products = [
    'Laptop Pro', 'Smartphone X', 'Wireless Headphones',
    'Smartwatch Ultra', '4K Monitor', 'Tablet Air',
    'Mechanical Keyboard', 'Ergonomic Chair', 'USB-C Dock', 'Webcam HD'
]
regions = ['North America', 'Europe', 'Asia-Pacific', 'Latin America']

# Generate 100 sales records
n_records = 100
sample_products = np.random.choice(products, size=n_records)
sample_regions = np.random.choice(regions, size=n_records)
sample_quantities = np.random.randint(1, 15, size=n_records)

# Base unit prices per product
price_map = {
    'Laptop Pro': 1200.0,
    'Smartphone X': 850.0,
    '4K Monitor': 450.0,
    'Tablet Air': 500.0,
    'Smartwatch Ultra': 300.0,
    'Ergonomic Chair': 250.0,
    'USB-C Dock': 130.0,
    'Wireless Headphones': 120.0,
    'Mechanical Keyboard': 90.0,
    'Webcam HD': 70.0
}

# Calculate revenue with minor variance (promotions / volume pricing)
base_prices = np.array([price_map[p] for p in sample_products])
discounts = np.random.uniform(0.90, 1.05, size=n_records)
revenues = np.round(sample_quantities * base_prices * discounts, 2)

# Create DataFrame
df = pd.DataFrame({
    'product': sample_products,
    'region': sample_regions,
    'quantity': sample_quantities,
    'revenue': revenues
})

# Save dataset to CSV
df.to_csv('sales_data.csv', index=False)

# Aggregate metrics by product
product_summary = df.groupby('product').agg(
    total_revenue=('revenue', 'sum'),
    total_quantity=('quantity', 'sum'),
    order_count=('product', 'count'),
    avg_order_value=('revenue', 'mean')
).sort_values(by='total_revenue', ascending=False)

# Identify Top 5 products by revenue
top_5_products = product_summary.head(5)

# Regional breakdown for the top 5 products
top_5_names = top_5_products.index.tolist()
regional_breakdown = df[df['product'].isin(top_5_names)].pivot_table(
    index='product',
    columns='region',
    values='revenue',
    aggfunc='sum',
    fill_value=0
).loc[top_5_names]

# Visualization
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Subplot 1: Top 5 Products by Revenue
bars = ax1.barh(top_5_products.index[::-1], top_5_products['total_revenue'][::-1] / 1000, color='#1f77b4', edgecolor='black')
ax1.set_xlabel('Total Revenue (in Thousands USD)', fontsize=11, fontweight='bold')
ax1.set_title('Top 5 Products by Total Revenue', fontsize=13, fontweight='bold', pad=12)
for bar in bars:
    width = bar.get_width()
    ax1.text(width + 0.5, bar.get_y() + bar.get_height()/2, f'${width:,.1f}K', ha='left', va='center', fontsize=9, fontweight='bold')

# Subplot 2: Regional Revenue Breakdown
regional_breakdown.plot(kind='bar', ax=ax2, colormap='viridis', edgecolor='black')
ax2.set_xlabel('Product', fontsize=11, fontweight='bold')
ax2.set_ylabel('Revenue (USD)', fontsize=11, fontweight='bold')
ax2.set_title('Top 5 Products Revenue by Region', fontsize=13, fontweight='bold', pad=12)
ax2.legend(title='Region', fontsize=9)
ax2.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('top_5_products_analysis.png', dpi=300)
plt.close()
```

---

### 2. Output and Results

#### Sample Dataset (First 5 Rows)
```
               product         region  quantity   revenue
0  Mechanical Keyboard  Latin America         1     82.26
1     Smartwatch Ultra         Europe         3    931.12
2      Ergonomic Chair  North America        10   2587.66
3           4K Monitor  Latin America        12   5372.81
4  Mechanical Keyboard   Asia-Pacific         8    684.62
```

#### Top 5 Products by Total Revenue

| Rank | Product | Total Revenue (USD) | Total Quantity Sold | Order Count | Avg Order Value (USD) |
|---|---|---|---|---|---|
| **1** | **Laptop Pro** | **$63,941.38** | 54 | 7 | $9,134.48 |
| **2** | **Smartphone X** | **$57,295.79** | 70 | 10 | $5,729.58 |
| **3** | **4K Monitor** | **$30,484.94** | 70 | 10 | $3,048.49 |
| **4** | **Ergonomic Chair** | **$26,600.18** | 108 | 15 | $1,773.35 |
| **5** | **Smartwatch Ultra** | **$21,155.92** | 71 | 9 | $2,350.66 |

#### Regional Revenue Breakdown for Top 5 Products

| Product | Asia-Pacific | Europe | Latin America | North America | Total |
|---|---|---|---|---|---|
| **Laptop Pro** | $26,621.25 | $8,378.95 | $0.00 | $28,941.18 | $63,941.38 |
| **Smartphone X** | $19,906.66 | $16,858.71 | $14,124.03 | $6,406.39 | $57,295.79 |
| **4K Monitor** | $7,436.83 | $10,127.95 | $6,788.30 | $6,131.86 | $30,484.94 |
| **Ergonomic Chair** | $0.00 | $5,099.74 | $3,938.36 | $17,562.08 | $26,600.18 |
| **Smartwatch Ultra** | $6,562.85 | $931.12 | $5,285.29 | $8,376.66 | $21,155.92 |

---

### 3. Key Insights

1. **High Unit-Value Driver**: **Laptop Pro** is the top revenue generator ($63,941.38) despite having fewer orders (7 orders, 54 units), driven by its high average order value ($9,134.48).
2. **Volume Leader**: **Ergonomic Chair** achieved the highest unit volume (108 units across 15 transactions), contributing $26,600.18 in revenue.
3. **Geographic Distribution**:
   - **North America** and **Asia-Pacific** accounted for the vast majority of **Laptop Pro** sales ($28.9K and $26.6K respectively).
   - **Smartphone X** demonstrated balanced demand across all four regions, led by Asia-Pacific ($19.9K) and Europe ($16.9K).

---

### 4. Generated Artifacts
- **Dataset File**: `sales_data.csv` (100 rows containing `product`, `region`, `quantity`, `revenue`)
- **Visualization File**: `top_5_products_analysis.png` (high-resolution dual-panel plot with horizontal revenue ranking and regional performance breakdown)

### Creating an agent with pre-loaded data

You can define the agent's environment with sources, so data is ready before the agent starts:

In [73]:
# Create an agent with GCS sources pre-loaded using the SDK.
my_slides_agent = client.agents.create(
    id=f"my-gemini-api-agent-{UNIQUE_SUFFIX}",
    base_agent=AGENT,
    system_instruction=(
        "You are a software engineer speciliazed in the Gemini API. "
        "Use the skills available in /.agents/skills/ to create amazing apps."
    ),
    base_environment={
        "type": "remote",
        "sources": [
            {
                "type": "repository",
                "source": "https://github.com/google-gemini/gemini-skills",
                "target": "/.agents/skills",
            }
        ],
    },
)

print(f"✓ Agent created: {my_slides_agent.id}")

✓ Agent created: my-gemini-api-agent-915849b1


In [74]:
# Invoke the custom agent.
interaction = client.interactions.create(
    agent=f"my-gemini-api-agent-{UNIQUE_SUFFIX}",
    input="Tell me what you can do with your skills?",
    environment="remote",
)

Markdown(interaction.output_text)

I specialize in building applications using the Gemini platform and Google GenAI ecosystem. Equipped with specialized skills, I can architect, code, and deploy end-to-end solutions across text, multimodal reasoning, real-time live streaming, and generative video.

Here is a breakdown of what I can build for you:

---

### 1. Gemini API & Agent Development (`gemini-api-dev`)
Using the modern **Interactions API** and current SDKs (`google-genai` for Python and `@google/genai` for TypeScript/JavaScript), I build standard and complex agentic workflows:

* **Text & Complex Reasoning**:
  * Build on the latest models: `gemini-3.8-flash`, `gemini-3.5-flash-lite`, and `gemini-3.1-pro-preview`.
  * Support for Chain-of-Thought thinking with verifiable thought signatures and structured schema outputs.
* **Multimodal Generation & Understanding**:
  * High-fidelity image generation and editing using Nano Banana models (`gemini-3-pro-image`, `gemini-3.1-flash-image`, `gemini-3.1-flash-lite-image`).
  * Speech synthesis using `gemini-3.1-flash-tts-preview` with Director’s Chair expressive prompting.
  * Speech-to-text using `gemini-3.5-transcribe` (supporting verbatim and smart cleanup modes).
  * Video, audio, and large-document ingestion, caching, and multimodal embeddings (`gemini-embedding-2`).
* **Autonomous & Managed Agents**:
  * **Antigravity Agent (`antigravity-preview-09-2026`)**: Deploy managed sandbox environments capable of running code (Bash, Python, Node.js), managing files, and browsing the web.
  * **Deep Research Agents (`deep-research-preview-04-2026` & `deep-research-max-preview-04-2026`)**: Autonomous, multi-step web research that explores sources in background sessions and produces comprehensive research briefs.
  * **Custom Managed Agents**: Define custom agent roles, provision sandbox repositories, and configure agent hooks.
* **Tool Calling & Streaming**:
  * Real-time Server-Sent Events (SSE) streaming with granular step-level deltas (`interaction.created`, `step.delta`, `interaction.completed`).
  * Integration with Google Search grounding, Code Execution, URL context, File Search, and MCP (Model Context Protocol) servers.
* **Architecture Modernization**:
  * Migrate legacy code from deprecated `generateContent` or legacy SDKs (`google-generativeai`, `@google/generative-ai`) to the unified Interactions API.

---

### 2. Real-Time Gemini Live API (`gemini-live-api-dev`)
For low-latency, bidirectional, multimodal voice and video applications over WebSockets:

* **Ultra-Low Latency Voice & Video Agents**:
  * **`gemini-3.8-live`**: Real-time conversational agent capable of simultaneous mic audio and camera/screen frame streaming (16kHz PCM audio in, 24kHz audio out).
  * **`gemini-3.8-live-extended-thinking`**: Conversational agent with deep background reasoning that speaks natural conversational fillers (*"Checking flight details for you now..."*) while asynchronously executing non-blocking tools (`behavior: "NON_BLOCKING"`).
* **Live Streaming Transcription & Translation**:
  * **`gemini-3.5-transcribe-live`**: Real-time streaming speech-to-text with interim hypotheses and finalized outputs in verbatim or smart modes.
  * **`gemini-3.5-live-translate-preview`**: Speech-to-speech simultaneous live translation across 70+ languages.
* **Advanced Session Control**:
  * Voice Activity Detection (VAD), client-side Hybrid VAD (`audio_stream_end`), and push-to-talk controls.
  * Natural mid-sentence interruption handling with immediate audio queue flushing.
  * Full session mid-stream context injection via `send_client_content`.
  * Ephemeral token generation for secure client-side browser and mobile deployments without leaking API keys.
  * Integration with real-time stacks like LiveKit, Pipecat, Fishjam, and Firebase AI Logic.

---

### 3. Generative Video with Gemini Omni Flash (`gemini-omni-flash-api`)
Leveraging `gemini-omni-1.1-flash` alongside automated tools and `ffmpeg` workflows for video creation and editing:

* **Text-to-Video & Camera Control**:
  * Generate videos up to 4K resolution (`360p`, `720p`, `1080p`, `4k`) in 16:9 landscape or 9:16 portrait formats, with natural dialogue, sound effects, or background scores.
* **First-Frame & Frame Interpolation**:
  * Animate a starting frame (`<FIRST_FRAME>`).
  * Generate seamless transitions between a starting frame and an ending frame (`<FIRST_FRAME>` to `<LAST_FRAME>`).
  * Create smooth looping videos by binding the same image to both boundaries.
* **Video Extensions**:
  * Extend existing videos in 10-second increments up to a total length of 40 seconds, maintaining continuous motion, characters, and audio.
* **Reference-Guided Generation**:
  * Steer character design, visual styles, or motion dynamics using reference images (`<IMAGE_REF_N>`) and reference videos (`<VIDEO_REF_N>`).
* **Turn-by-Turn Video Editing**:
  * Style transfer, inpainting, and object modification on existing videos.
  * Audio control: preserve existing audio or completely strip the soundtrack (`--strip-audio`) to let the model generate new audio from scratch.
* **Batch Production**:
  * Automated CLI tooling for media preprocessing (`prep_video.py`), video inspection (`inspect_video.py`), and concurrent batch jobs (`generate_video.py --batch jobs.json`).

---

### What would you like to build?
Whether you need an interactive voice assistant, an autonomous research workflow, a generative video pipeline, or a full-stack Next.js/Python application, let me know what you'd like to create!

### Forking from an existing environment

If you've already set up a sandbox you like (installed packages, created files, etc.), you can fork it into a new agent using the `environment_id` from a previous interaction:

```python
my_forked_agent = client.agents.create(
    id="my-forked-agent",
    base_agent=AGENT,
    system_instruction="Your custom instructions here.",
    base_environment={"env_id": "YOUR_ENVIRONMENT_ID"},
)
```

This captures the exact state of that sandbox — all installed packages, files, and configuration.

In [78]:
my_forked_agent = client.agents.create(
    id=f"my-forked-agent-{UNIQUE_SUFFIX}",
    base_agent=AGENT,
    system_instruction="I want all your apps to use the Live API",
    base_environment={"env_id": interaction.environment_id},
)
print(f"✓ Agent forked: {my_forked_agent.id}")

✓ Agent forked: my-forked-agent-915849b1


### Managing agents (CRUD)

The `/agents` endpoint supports full lifecycle management:

In [80]:
# List all your agents.
print("Your agents:")
for agent in client.agents.list().agents:
    print(f"- {agent.id}")

# Get a specific agent's details.
agent = client.agents.get(id=f"my-data-analyst-{UNIQUE_SUFFIX}")
print(f"\nAgent details for {agent.id}:")
print(f"Base agent: {agent.base_agent}")
print(f"System instruction: {agent.system_instruction}")

Your agents:
- my-data-analyst-71b64eaf
- my-data-analyst-915849b1
- my-data-analyst-b748d068
- my-forked-agent-915849b1
- my-gemini-api-agent-71b64eaf
- my-gemini-api-agent-915849b1
- my-gemini-api-agent-b748d068

Agent details for my-data-analyst-915849b1:
Base agent: antigravity-preview-05-2026
System instruction: You are a data analysis assistant. Always write Python code using pandas to answer questions. Show your code and output clearly. When creating visualizations, save them as PNG files.


In [81]:
# Clean up: delete the agents you created.
for agent_name in ["my-data-analyst", "my-forked-agent", "my-gemini-api-agent"]:
    try:
        client.agents.delete(id=agent_name)
        print(f"✓ Deleted {agent_name}")
    except Exception as e:
        print(f"  Failed to delete {agent_name}: {e}")

✓ Deleted my-data-analyst
✓ Deleted my-forked-agent
✓ Deleted my-gemini-api-agent


### Agent directory structure

Behind the API, an agent is defined by a simple set of files. This is what gets deployed when you create one:

```
my-agent/
├── agent.yaml       # Configuration: base agent, tools, environment
├── AGENTS.md        # System instructions (loaded automatically)
├── skills/          # Custom SKILL.md files that extend capabilities
└── workspace/       # Files seeded into the remote sandbox at startup
```

- **`agent.yaml`** maps directly to the `/agents` API resource
- **`AGENTS.md`** provides system instructions — automatically loaded by the harness
- **`skills/`** contains specialized `SKILL.md` files the agent discovers and uses
- **`workspace/`** files are injected into the sandbox at startup

This file-based structure makes agents easy to version-control, share, and iterate on. Check the [documentation](https://ai.google.dev/gemini-api/docs/custom-agents#file-based_customization) for more details.

## 6. Streaming

For longer tasks, enable streaming with `stream=True` to get real-time updates as the agent works. Instead of waiting for the complete response, you receive a stream of **Server-Sent Events (SSE)** that let you show progress to the user.

### Event types

The stream delivers events that tell you what the agent is doing:

| Event type | Meaning | What to do |
|------------|---------|------------|
| `interaction.created` | The interaction was created | Store the `id` for later reference |
| `interaction.status_update` | Status changed (e.g., `in_progress`) | Update UI status indicator |
| `step.start` | A new step began (thinking, tool call, output) | Show a loading indicator |
| `step.delta` | Incremental content — a chunk of text, thought, or tool output | **Append to display** — this is the main content |
| `step.stop` | A step completed | Hide loading indicator |
| `interaction.completed` | The agent finished all work | Finalize the UI |

The `step.delta` events are where the content lives. Each delta has a `type` (e.g., `text`, `thought`, `function_call`, `function_result`) and content you can render incrementally.

In [82]:
# Stream a response and collect the text as it arrives.
stream = client.interactions.create(
    agent=AGENT,
    input="Write a short poem about the ocean.",
    stream=True,
    environment="remote",
)

collected_text = []

for event in stream:
    # Show the event type so you can see the lifecycle.
    if event.event_type in ("interaction.created", "step.start", "step.stop", "interaction.completed"):
        print(f"[{event.event_type}]")

    # step.delta events carry the actual content.
    elif event.event_type == "step.delta":
        delta = event.delta
        if hasattr(delta, "text") and delta.text:
            print(delta.text, end="", flush=True)
            collected_text.append(delta.text)

print(f"\n\n--- Collected {len(collected_text)} text chunks ---")

[interaction.created]
[step.start]
A boundless cradle deep and wide,  
Where silver currents dance and glide.  
In rhythmic pulse of ebb and flow,  
The ancient tides drift soft and slow.  

Beneath the sunlit turquoise crest,  
A silent world is rocked to rest,  
Whispering secrets to the shore,  
In timeless waves forevermore.[step.stop]
[interaction.completed]


--- Collected 4 text chunks ---


## 7. Advanced features

### Network configuration

By default, the agent's sandbox has unrestricted outbound network access. You can control this with the `network` field:
- **Allowlist specific domains** — only requests to listed domains are permitted
- **Inject credentials** — automatically add headers (API keys, tokens) to outbound requests
- **Disable network** — set `network: "disabled"` to block all outbound traffic

In [83]:
# Allow the agent to call only the Gemini API, with an auto-injected API key.
interaction = client.interactions.create(
    agent=AGENT,
    input="Use curl to call the Gemini API and list available models. Show the first 3.",
    environment={
        "type": "remote",
        "network": {
            "allowlist": [
                {
                    "domain": "generativelanguage.googleapis.com",
                    "transform": [{"x-goog-api-key": GEMINI_API_KEY}],
                },
            ]
        },
    },
)

Markdown(interaction.output_text)

To list available models using `curl` and display the first three results, run:

```bash
curl -s https://generativelanguage.googleapis.com/v1beta/models | jq '.models[:3]'
```

### Output

```json
[
  {
    "name": "models/gemini-2.5-flash",
    "version": "001",
    "displayName": "Gemini 2.5 Flash",
    "description": "Stable version of Gemini 2.5 Flash, our mid-size multimodal model that supports up to 1 million tokens, released in June of 2025.",
    "inputTokenLimit": 1048576,
    "outputTokenLimit": 65536,
    "supportedGenerationMethods": [
      "generateContent",
      "countTokens",
      "createCachedContent",
      "batchGenerateContent"
    ],
    "temperature": 1,
    "topP": 0.95,
    "topK": 64,
    "maxTemperature": 2,
    "thinking": true
  },
  {
    "name": "models/gemini-2.5-pro",
    "version": "2.5",
    "displayName": "Gemini 2.5 Pro",
    "description": "Stable release (June 17th, 2025) of Gemini 2.5 Pro",
    "inputTokenLimit": 1048576,
    "outputTokenLimit": 65536,
    "supportedGenerationMethods": [
      "generateContent",
      "countTokens",
      "createCachedContent",
      "batchGenerateContent"
    ],
    "temperature": 1,
    "topP": 0.95,
    "topK": 64,
    "maxTemperature": 2,
    "thinking": true
  },
  {
    "name": "models/gemini-2.5-flash-preview-tts",
    "version": "gemini-2.5-flash-exp-tts-2025-05-19",
    "displayName": "Gemini 2.5 Flash Preview TTS",
    "description": "Gemini 2.5 Flash Preview TTS",
    "inputTokenLimit": 8192,
    "outputTokenLimit": 16384,
    "supportedGenerationMethods": [
      "countTokens",
      "generateContent"
    ],
    "temperature": 1,
    "topP": 0.95,
    "topK": 64,
    "maxTemperature": 2
  }
]
```

### Download environment snapshots

You can download all the files the agent created or modified as a tar archive. This lets you retrieve the agent's work products — code, data, reports — from the sandbox.

In [84]:
import subprocess
import tarfile
import os

# Create an interaction where the agent produces files.
interaction = client.interactions.create(
    agent=AGENT,
    input=(
        "Create a directory called 'project' with a README.md and a hello.py script. "
        "List the files you created."
    ),
    environment="remote",
)

env_id = interaction.environment_id
print(f"Environment ID: {env_id}")

Markdown(interaction.output_text)

Environment ID: 26a632717576cab2000e9d9a2a714a95


Created the `project` directory with the requested files:

- `README.md`
- `hello.py`

In [85]:
# Download the environment snapshot.
download_url = (
    f"https://generativelanguage.googleapis.com/v1beta/"
    f"files/environment-{env_id}:download?alt=media"
)

result = subprocess.run(
    ["curl", "-L", "-s", "-o", "snapshot.tar",
     "-H", f"x-goog-api-key: {GEMINI_API_KEY}",
     download_url],
    capture_output=True, text=True,
)

if os.path.exists("snapshot.tar") and os.path.getsize("snapshot.tar") > 0:
    with tarfile.open("snapshot.tar") as tar:
        print("Files in snapshot:")
        for member in tar.getmembers():
            print(f"  {member.name} ({member.size} bytes)")
else:
    print("Snapshot not available (environment may have expired).")

Files in snapshot:
  . (0 bytes)
  ./project (0 bytes)
  ./project/README.md (74 bytes)
  ./project/hello.py (78 bytes)


## Next steps

You've walked through the core capabilities of managed agents:

1. ✅ **Simple Q&A** — the agent can answer questions like an LLM
2. ✅ **Multi-turn** — persistent sandbox enables stateful conversations
3. ✅ **Built-in tools** — code execution, web search, file management
4. ✅ **Data loading** — inject files via inline, GCS, or GitHub sources
5. ✅ **Custom agents** — reusable configurations with instructions, skills, and environment
6. ✅ **Streaming** — real-time updates as the agent works
7. ✅ **Advanced** — network control and environment snapshots

### Learn more

- **[Getting Started notebook](./Get_started_interactions_api.ipynb)** — the `model=`-based Interactions API for standard generation, multi-turn, and tools
- **[Managed Agents documentation](https://ai.google.dev/gemini-api/docs/eap/gemini-agents/gemini-agents)** — full reference for the agent API
- **[Code Execution](./Code_Execution.ipynb)** — model-based code execution
- **[Search Grounding](./Search_Grounding.ipynb)** — model-based web search
- **[Function Calling](./Function_calling.ipynb)** — custom function declarations